<a href="https://colab.research.google.com/github/felipetavaressilvaoliveira-netizen/senacai/blob/main/c%C3%B3digo_para_uso_token_BV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import tiktoken

# 1. Definir o encoding (o GPT-4 e GPT-3.5 usam o cl100k_base)
encoding = tiktoken.get_encoding("cl100k_base")

text = "Engenharia de Dados com IA Generativa"

# 2. Transformar a string em uma lista de IDs de tokens
tokens_inteiros = encoding.encode(text)

# 3. Contar o tamanho da lista
quantidade_tokens = len(tokens_inteiros)

print(f"Texto: '{text}'")
print(f"Tokens (IDs): {tokens_inteiros}")
print(f"Quantidade total de tokens: {quantidade_tokens}")

# Bônus: Mostrar a tradução de cada ID de volta para string
print("Pedaços de texto por token:", [encoding.decode_single_token_bytes(token) for token in tokens_inteiros])

Texto: 'Engenharia de Dados com IA Generativa'
Tokens (IDs): [4198, 40967, 10649, 409, 423, 5670, 470, 44190, 2672, 28952]
Quantidade total de tokens: 10
Pedaços de texto por token: [b'Eng', b'enh', b'aria', b' de', b' D', b'ados', b' com', b' IA', b' Gener', b'ativa']


In [5]:
import tiktoken
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 1. Setup inicial (necessário apenas na primeira execução)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Adicionado para corrigir o LookupError

def clean_and_count(text):
    # 2. Definir o idioma das stop words (Português)
    stop_words = set(stopwords.words('portuguese'))

    # 3. Tokenização simples por palavras para limpeza
    words = word_tokenize(text)

    # 4. Remoção de redundâncias (stop words e pontuação)
    filtered_text = [w for w in words if w.lower() not in stop_words and w.isalnum()]
    clean_string = " ".join(filtered_text)

    # 5. Contagem de tokens reais para a LLM (tiktoken)
    encoding = tiktoken.get_encoding("cl100k_base")
    raw_tokens = len(encoding.encode(text))
    clean_tokens = len(encoding.encode(clean_string))

    print(f"Texto Original: {text}")
    print(f"Texto Limpo: {clean_string}")
    print(f"Tokens Originais: {raw_tokens}")
    print(f"Tokens após Limpeza: {clean_tokens}")
    print(f"Economia: {raw_tokens - clean_tokens} tokens ({(1 - clean_tokens/raw_tokens)*100:.1f}%)的发展)")

# Exemplo prático
frase = "A engenharia de dados é fundamental para o sucesso da inteligência artificial"
clean_and_count(frase)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Texto Original: A engenharia de dados é fundamental para o sucesso da inteligência artificial
Texto Limpo: engenharia dados fundamental sucesso inteligência artificial
Tokens Originais: 15
Tokens após Limpeza: 9
Economia: 6 tokens (40.0%)的发展)


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
# A mágica do RAG: Injetando dados do DW no prompt
dados_dw = "Vendas Janeiro: R$ 1M. Vendas Fevereiro: R$ 1.2M." # Resultado de uma Query
pergunta_usuario = "Qual o crescimento entre Jan e Fev?"

# O prompt que consome tokens de forma inteligente
prompt_final = f"Com base nestes dados: {dados_dw}. Responda: {pergunta_usuario}"

# Aqui o Engenheiro de Dados controla o custo/contexto
tokens_gastos = len(encoding.encode(prompt_final))
print(tokens_gastos)

44


In [10]:
import tiktoken
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Setup de pacotes (necessário para a lógica de limpeza)
nltk.download('punkt')
nltk.download('stopwords')

def analise_tokens_corporativo():
    encoding = tiktoken.get_encoding("cl100k_base")
    stop_words = set(stopwords.words('portuguese'))

    # --- SIMULAÇÃO DE DADOS DO DW ---
    # Imagine que isso veio de um SELECT sum(vendas) no seu Snowflake/BigQuery
    dados_dw = "Resultado da query: Vendas Janeiro: R$ 1.000.000. Vendas Fevereiro: R$ 1.250.000. Status: Crescimento detectado."
    pergunta_usuario = "Qual foi o percentual de crescimento entre janeiro e fevereiro?"

    # --- LIMPEZA (OPCIONAL MAS RECOMENDADA) ---
    def limpar(texto):
        palavras = word_tokenize(texto)
        return " ".join([w for w in palavras if w.lower() not in stop_words and w.isalnum()])

    # --- CÁLCULO DE TOKENS INDIVIDUAIS ---
    tokens_dw = len(encoding.encode(dados_dw))
    tokens_pergunta = len(encoding.encode(pergunta_usuario))

    # --- CONSTRUÇÃO DO PROMPT (ORQUESTRAÇÃO) ---
    # O "Template" também consome tokens!
    prompt_final = f"Instrução: Atue como analista de dados. Baseado nos dados: {dados_dw}. Responda: {pergunta_usuario}"
    tokens_prompt_total = len(encoding.encode(prompt_final))

    # --- OUTPUT TÉCNICO ---
    print(f"--- ANÁLISE DE VOLUMETRIA ---")
    print(f"1. Tokens Dados DW:      {tokens_dw}")
    print(f"2. Tokens Pergunta:      {tokens_pergunta}")
    print(f"-----------------------------")
    print(f"Soma Simples (1+2):      {tokens_dw + tokens_pergunta}")
    print(f"Total Prompt Final:      {tokens_prompt_total}")
    print(f"Overhead do Template:    {tokens_prompt_total - (tokens_dw + tokens_pergunta)} tokens")

    print(f"\n--- PROMPT ENVIADO À LLM ---\n{prompt_final}")

analise_tokens_corporativo()

--- ANÁLISE DE VOLUMETRIA ---
1. Tokens Dados DW:      40
2. Tokens Pergunta:      16
-----------------------------
Soma Simples (1+2):      56
Total Prompt Final:      77
Overhead do Template:    21 tokens

--- PROMPT ENVIADO À LLM ---
Instrução: Atue como analista de dados. Baseado nos dados: Resultado da query: Vendas Janeiro: R$ 1.000.000. Vendas Fevereiro: R$ 1.250.000. Status: Crescimento detectado.. Responda: Qual foi o percentual de crescimento entre janeiro e fevereiro?


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
